In [ ]:
import torch
import os
import torch.nn.functional as F
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer,AutoModel

In [ ]:
# 1. Configuration
TEACHER_ID = "openai/clip-vit-base-patch32"
STUDENT_ID = "Shushant/nepaliBERT"
#STUDENT_ID = "IRIISNEPAL/RoBERTa_Nepali_110M"
MAX_LENGTH = 77  # CLIP's hard limit
BATCH_SIZE = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
LOCAL_MODEL_PATH="/content/models"

#Download ImageNet

In [ ]:
# 2. Path to your file (update 'My Drive/folder_name/' to where your file is)
tar_path = '/content/drive/My Drive/ILSVRC2012_img_val.tar'

In [ ]:
!cp "$tar_path" .

In [ ]:
# Create a folder and extract there
!mkdir imagenet_val
!tar -xf ILSVRC2012_img_val.tar -C ./imagenet_val

In [ ]:
devkit_tar_path = '/content/drive/My Drive/ILSVRC2012_devkit_t12.tar.gz'

In [ ]:
!cp "$devkit_tar_path" .

In [ ]:
# Create a folder and extract there
!mkdir imagenet_val_dev
!tar -xf ILSVRC2012_devkit_t12.tar.gz -C ./imagenet_val_dev

In [ ]:
gt_file = "imagenet_val_dev/ILSVRC2012_devkit_t12/data/ILSVRC2012_validation_ground_truth.txt"
mat_file="imagenet_val_dev/ILSVRC2012_devkit_t12/data/meta.mat"
img_folder = "imagenet_val"


In [ ]:
import scipy.io

# 1. Load the metadata
meta = scipy.io.loadmat(mat_file, squeeze_me=True)
synsets = meta['synsets']

# 2. Extract the 1,000 target classes (IDs 1 to 1000)
low_level = []
for s in synsets:
    if s[0] <= 1000:
        low_level.append({'ID': int(s[0]), 'WNID': s[1], 'Label': s[2]})

# 3. Sort them alphabetically by WNID to get the 0-999 order
sorted_by_wnid = sorted(low_level, key=lambda x: x['WNID'])

# 4. Create Conversion Dictionaries
# From Ground Truth ID (1-1000) -> Model Index (0-999)
id_to_model_idx = {item['ID']: i for i, item in enumerate(sorted_by_wnid)}

# From Model Index (0-999) -> Ground Truth ID (1-1000)
model_idx_to_id = {i: item['ID'] for i, item in enumerate(sorted_by_wnid)}

# Example usage:
print(f"Sea Snake (ID 490) maps to Model Index: {id_to_model_idx[490]}")
# Output: 65

In [ ]:
import json
import requests

# Link to the raw JSON file
url = "https://raw.githubusercontent.com/gregor-ge/Babel-ImageNet/main/data/babel_imagenet.json"
data = requests.get(url).json()

# Extract Nepali subset
# The structure is: { "lang_code": [indices, labels] }
nep_indices, nep_labels = data["NE"]

# Create a mapping dictionary {ImageNet_Index: "Nepali_Label"}
idx_to_nepali = {idx: label for idx, label in zip(nep_indices, nep_labels)}
imagenet_idx_to_nepali = {model_idx_to_id[idx]: label for idx, label in zip(nep_indices, nep_labels)}

print(f"Loaded {len(nep_labels)} Nepali labels.")

print(f"Original Index of Nepali Conversion with ImageNet: Index")
top5 = list(idx_to_nepali.items())[:5]
for k, v in top5:
    print(k, v)

print(f"Mapped Index of Nepali Conversion with ImageNet: Index")
top5 = list(imagenet_idx_to_nepali.items())[:5]
for k, v in top5:
    print(k, v)

In [ ]:
supported_indices = indices = imagenet_idx_to_nepali.keys()

In [ ]:
import os
from collections import Counter
# 1. Setup paths

# 2. Get sorted list of image filenames
images = sorted([f for f in os.listdir(img_folder) if f.lower().endswith(('.jpeg', '.jpg', '.JPEG'))])

# 3. Read the ground truth labels (Official 1-1000 -> Model 0-999)
with open(gt_file, 'r') as f:
    labels = [int(line.strip()) for line in f.readlines()]

# 4. Filter and Map (Updated to include full paths)
all_matches = {
    img: {
        "idx": lbl,
        "label_en": sorted_by_wnid[id_to_model_idx[lbl]]["Label"]
    }
    for img, lbl in zip(images, labels)
}

In [ ]:
filtered_matches = {
    img: {
        "idx": info["idx"],
        "label_ne": imagenet_idx_to_nepali[info["idx"]],
        "label_en": info["label_en"],
        "path": os.path.abspath(os.path.join(img_folder, img))
    }
    for img, info in all_matches.items()
    if info["idx"] in supported_indices
}

# Count occurrences of each Nepali label
label_counts = Counter([val['label_ne'] for val in filtered_matches.values()])

# 5. Results
print("--- Dataset Statistics ---")
print(f"Total images in folder:      {len(images)}")
print(f"Total matched to GT:         {len(all_matches)}")
print(f"Total filtered (Nepali):     {len(filtered_matches)}")
print(f"Unique Nepali Classes:       {len(label_counts)}")
print(f"Percentage kept:             {(len(filtered_matches)/len(all_matches))*100:.2f}%")

# Displaying label counts for the top classes
print("\n--- Label Counts (Top 5) ---")
for label, count in label_counts.most_common(5):
    print(f"{label}: {count} images")

# Quick verification
if filtered_matches:
    first_img = list(filtered_matches.keys())[1]
    sample = filtered_matches[first_img]
    print(f"\nSample Match:")
    print(f"Image Name: {first_img}")
    print(f"Full Path:  {sample['path']}")
    print(f"Index:      {sample['idx']}")
    print(f"English:    {sample['label_en']}")
    print(f"Nepali:     {sample['label_ne']}")

In [ ]:
first_img = list(filtered_matches.keys())[6]
sample = filtered_matches[first_img]
print(f"\nSample Match:")
print(f"Image Name: {first_img}")
print(f"Full Path:  {sample['path']}")
print(f"Index:      {sample['idx']}")
print(f"English:    {sample['label_en']}")
print(f"Nepali:     {sample['label_ne']}")

In [ ]:
imagenet_templates_nepali = [
    'एउटा नराम्रो {} को फोटो।',
    'धेरै {} हरूको फोटो।',
    '{} को एउटा मूर्ति।',
    'मुस्किलले देखिने {} को फोटो।',
    '{} को एउटा कम रिजोलुसनको फोटो।',
    '{} को एउटा रेन्डरिङ।',
    '{} को भित्ते चित्र।',
    '{} को ग्राफिटी।',  # Extracted
    'त्यो {} को एउटा नराम्रो फोटो।',
    'काटिएको {} को फोटो।',
    'क्रप गरिएको {} को फोटो।',  # Extracted
    '{} को एउटा ट्याटु।',
    'कढाई गरिएको {}।',
    'देख्न गाह्रो हुने एउटा {} को फोटो।',
    'एउटा चम्किलो {} को फोटो।',
    'एउटा सफा {} को फोटो।',
    'एउटा फोहोर {} को फोटो।',
    'त्यो {} को एउटा अँध्यारो फोटो।',
    '{} को एउटा चित्र।',
    '{} को एउटा ड्रइङ।',  # Extracted
    'मेरो {} को एउटा फोटो।',
    'प्लास्टिकको {}।',
    'राम्रो देखिने {} को फोटो।',
    'एउटा नजिकैबाट खिचिएको {} को फोटो।',
    'एउटा क्लोज-अप {} को फोटो।',  # Extracted
    'त्यो {} को श्याम-श्वेत फोटो।',
    'त्यो {} को ब्ल्याक एण्ड ह्वाइट फोटो।',  # Extracted
    'त्यो {} को एउटा पेन्टिङ।',
    'एउटा {} को पेन्टिङ।',
    'त्यो {} को पिक्सेलेटेड फोटो।',
    'त्यो {} को एउटा मूर्ति।',
    'त्यो {} को चम्किलो फोटो।',
    'एउटा काटिएको {} को फोटो।',
    'एउटा क्रप गरिएको {} को फोटो।', # Extracted (Duplicate check: unique entry)
    'एउटा प्लास्टिकको {}।',
    'त्यो फोहोर {} को फोटो।',
    'बिग्रिएको {} को फोटो।',
    'जेपेग करप्टेड {} को फोटो।',  # Extracted
    'त्यो धमिलो {} को फोटो।',
    'त्यो {} को एउटा फोटो।',
    'त्यो {} को एउटा राम्रो फोटो।',
    'त्यो {} को एउटा रेन्डरिङ।',
    'भिडियो गेममा देखिने एउटा {}।',
    'एउटा मात्र {} को फोटो।',
    '{} को एउटा डूडल।',
    'त्यो {} को नजिकैबाट खिचिएको फोटो।',
    'त्यो {} को क्लोज-अप फोटो।',  # Extracted
    'एउटा {} को फोटो।',
    'ओरिगामी {}।',
    'कागजबाट बनेको {}।',  # Extracted
    'भिडियो गेममा देखिने त्यो {}।',
    '{} को एउटा स्केच।',
    'त्यो {} को एउटा डूडल।',
    'एउटा ओरिगामी {}।',
    'एउटा कम रिजोलुसनको {} को फोटो।',
    'खेलौना {}।',
    'त्यो {} को एउटा प्रस्तुति।',
    'त्यो {} को एउटा रेन्डिसन।',  # Extracted
    'त्यो सफा {} को फोटो।',
    'एउटा ठूलो {} को फोटो।',
    'एउटा {} को प्रस्तुति।',
    'एउटा {} को रेन्डिसन।',  # Extracted
    'एउटा राम्रो {} को फोटो।',
    'एउटा अनौठो {} को फोटो।',
    'एउटा धमिलो {} को फोटो।',
    'एउटा कार्टुन {}।',
    '{} को एउटा कला।',
    '{} को एउटा आर्ट।',  # Extracted
    'त्यो {} को एउटा स्केच।',
    'एउटा कढाई गरिएको {}।',
    'एउटा पिक्सेलेटेड {} को फोटो।',
    'मैले खिचेको त्यो {} को फोटो।',
    'त्यो बिग्रिएको {} को फोटो।',
    'त्यो जेपेग करप्टेड {} को फोटो।',  # Extracted
    'एउटा प्लसी {}।',
    'एउटा नरम कपडाको खेलौना {}।',  # Extracted
    'त्यो राम्रो {} को फोटो।',
    'त्यो सानो {} को फोटो।',
    'त्यो अनौठो {} को फोटो।',
    'त्यो कार्टुन {}।',
    'त्यो {} को कला।',
    'त्यो {} को आर्ट।',  # Extracted
    'त्यो {} को एउटा चित्र।',
    'त्यो {} को एउटा ड्रइङ।',  # Extracted
    'त्यो ठूलो {} को फोटो।',
    'एउटा श्याम-श्वेत {} को फोटो।',
    'एउटा ब्ल्याक एण्ड ह्वाइट {} को फोटो।',  # Extracted
    'त्यो प्लसी {}।',
    'त्यो नरम कपडाको खेलौना {}।',  # Extracted
    'एउटा अँध्यारो {} को फोटो।',
    'मैले खिचेको एउटा {} को फोटो।',
    'त्यो {} को भित्ते चित्र।',
    'त्यो {} को ग्राफिटी।',  # Extracted
    'एउटा खेलौना {}।',
    'मैले खिचेको मेरो {} को फोटो।',
    'एउटा सानो {} को फोटो।',
    'त्यो {} को एउटा ट्याटु।',
]

In [ ]:
image_paths = [info["path"] for info in filtered_matches.values()]
total_labels_ne   = [info["label_ne"] for info in filtered_matches.values()]
total_labels_en   = [info["label_en"] for info in filtered_matches.values()]

In [ ]:
# 1. Get your unique Nepali labels (The "Classes")
text_prompts = sorted(list(set(total_labels_ne)))
print(f"Number of unique classes: {len(text_prompts)}")

# 2. Create a mapping from Label String -> Integer Index
label_to_idx = {label: i for i, label in enumerate(text_prompts)}

# 3. Create your Ground Truth list
gt_labels = [label_to_idx[label] for label in total_labels_ne]

print(f"Length of gt_labels: {len(gt_labels)}")

#Download Models

In [ ]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from urllib.request import urlopen, Request
from tqdm.notebook import tqdm

def copy_worker(item_info):
    """Worker function to copy/download a single item only if it doesn't exist."""
    source, destination, item_name = item_info

    # Check if it already exists locally
    if os.path.exists(destination):
        return f"⏩ Skipped: {item_name} (Already exists)"

    # Handle Web URL download
    if source.startswith(("http://", "https://")):
        try:
            req = Request(source, headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(req) as response, open(destination, 'wb') as out_file:
                shutil.copyfileobj(response, out_file)
            return f"📥 URL Downloaded: {item_name}"
        except Exception as e:
            return f"❌ Failed URL: {item_name} ({e})"

    # Handle Google Drive / Local File system copy
    try:
        if os.path.isdir(source):
            shutil.copytree(source, destination)
            return f"✅ Folder copied: {item_name}"
        else:
            shutil.copy2(source, destination)
            return f"✅ File copied:   {item_name}"
    except Exception as e:
        return f"❌ Failed Copy: {item_name} ({e})"

def download_models_parallel(paths_to_download, local_path, max_workers=4):
    """Parallel downloader that accepts folders, single files, or file URLs."""
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)
        print(f"📁 Created local directory: {local_path}")

    tasks = []

    # Scenario A: User passed a string path (Drive/Local path or URL)
    if isinstance(paths_to_download, str):
        # 1. Check if it's a web URL
        if paths_to_download.startswith(("http://", "https://")):
            item_name = os.path.basename(paths_to_download.split("?")[0])
            dest_path = os.path.join(local_path, item_name)
            tasks.append((paths_to_download, dest_path, item_name))
            print(f"📦 Found 1 URL link. Processing...")

        # 2. Check if it's an existing path on Drive/Local
        elif os.path.exists(paths_to_download):
            # If it's a single file (like your distillation_epoch_4_step_150000.pt)
            if os.path.isfile(paths_to_download):
                item_name = os.path.basename(paths_to_download)
                dest_path = os.path.join(local_path, item_name)
                tasks.append((paths_to_download, dest_path, item_name))
                print(f"📦 Found 1 single file path. Processing...")

            # If it's a directory containing multiple models
            elif os.path.isdir(paths_to_download):
                items = os.listdir(paths_to_download)
                tasks = [(os.path.join(paths_to_download, i), os.path.join(local_path, i), i) for i in items]
                print(f"📦 Found folder with {len(items)} items. Processing...")
        else:
            print(f"❌ Error: Path '{paths_to_download}' not found.")
            return

    # Scenario B: User passed a list of mixed paths/URLs
    elif isinstance(paths_to_download, list):
        for item in paths_to_download:
            item_name = os.path.basename(item.split("?")[0])
            dest_path = os.path.join(local_path, item_name)
            tasks.append((item, dest_path, item_name))
        print(f"📦 Processing list of {len(tasks)} custom items...")

    if not tasks:
        print("Empty path list provided.")
        return

    # Using ThreadPoolExecutor with a progress bar
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(copy_worker, tasks), total=len(tasks)))

    # Print summary
    skipped = sum(1 for r in results if "Skipped" in r)
    success = sum(1 for r in results if "✅" in r or "📥" in r)
    failed = sum(1 for r in results if "❌" in r)

    print(f"\n✨ Done! {success} downloaded/copied, {skipped} skipped, {failed} failed.")

In [ ]:
model_path = '/content/drive/MyDrive/Nep-CLIP Datasets/Shushant-nepaliBERT distillation_epoch_4_step_150000.pt'
#model_path = '/content/drive/MyDrive/Nep-CLIP Datasets/RoBERTa_Nepali_110M distillation_epoch_4_step_150000.pt'

download_models_parallel(model_path,local_path=LOCAL_MODEL_PATH, max_workers=8)

In [ ]:
clip_processor = CLIPProcessor.from_pretrained(TEACHER_ID)
s_tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
clip_model = CLIPModel.from_pretrained(TEACHER_ID)

In [ ]:
class NepaliTextEncoder(nn.Module):
  def __init__(self, student_id):
      super().__init__()
      self.student = AutoModel.from_pretrained(student_id)
      self.student.eval()
      self.projection = nn.Linear(768, 512)

  def get_nepali_text_embeddings(self, student_inputs):
    text_embeddings = self.student(**student_inputs)
    return text_embeddings.pooler_output

  def forward(self,student_inputs):
    nepali_embeddings = self.get_nepali_text_embeddings(student_inputs=student_inputs)
    student_projected_embeddings = self.projection(nepali_embeddings)
    return  student_projected_embeddings

In [ ]:
import os
import torch

def get_all_models_with_loaded_instance(model_folder, STUDENT_ID):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # List all checkpoint files in folder 'x'
    checkpoint_files = [f for f in os.listdir(model_folder) if f.endswith(('.pt', '.pth'))]

    encoders_dict = {}

    for ckpt_name in checkpoint_files:
      print(f"Loading weights from: {ckpt_name}")
      ckpt_path = os.path.join(model_folder, ckpt_name)

      # 1. Initialize a fresh instance of your encoder
      encoder = NepaliTextEncoder(STUDENT_ID)

      # 2. Load the checkpoint
      ckpt = torch.load(ckpt_path, map_location=device)

      # 3. Load state dicts into the sub-modules
      encoder.student.load_state_dict(ckpt["student_state_dict"])
      encoder.projection.load_state_dict(ckpt["projection_state_dict"])

      # Store using the filename (or a pretty name) as the key
      encoders_dict[ckpt_name] = encoder

    return encoders_dict

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class ImagenetEvalDatasetWithTemplate(Dataset):
    def __init__(self, image_paths, labels, templates, image_processor, tokenizer, max_length=77):
        """
        Args:
            labels: The raw class names (e.g. ["dog", "cat"])
            templates: List of strings (e.g. ["a photo of a {}", "a crop of a {}"])
        """
        print(f"Pre-processing {len(image_paths)} images and {len(labels)} labels...")

        # 1. Pre-process Images
        processed_images = []
        for path in image_paths:
            img = Image.open(path).convert("RGB")
            encoding = image_processor(images=img, return_tensors="pt")
            processed_images.append(encoding.pixel_values.squeeze(0))
        self.pixel_values = torch.stack(processed_images, dim=0)

        # 2. Pre-tokenize Captions (Saving ALL keys)
        # We initialize a container for the stacked tensors
        self.text_inputs = {}
        temp_storage = {key: [] for key in tokenizer.model_input_names}

        for label in labels:
            templated_captions = [t.format(label) for t in templates]

            # Tokenize the batch of templates for this specific label
            tokens = tokenizer(
                templated_captions,
                padding="max_length",
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            # Store every key returned (input_ids, attention_mask, etc.)
            for key in tokens:
                temp_storage[key].append(tokens[key])

        # Stack everything into 3D tensors: [Num_Labels, Num_Templates, Max_Length]
        for key in temp_storage:
            self.text_inputs[key] = torch.stack(temp_storage[key], dim=0)

    def __len__(self):
        # Length is based on number of images for evaluation
        return len(self.pixel_values)

    def get_all_data(self):
        """Returns the image tensor and the dictionary of 3D text tensors"""
        return self.pixel_values, self.text_inputs

In [ ]:
# 1. Initialize Dataset
dataset_with_template = ImagenetEvalDatasetWithTemplate(
    image_paths=image_paths,
    labels=text_prompts,
    image_processor=clip_processor,
    tokenizer=s_tokenizer,
    templates=imagenet_templates_nepali, # Explicitly named
)
# 2. Get processed tensors
image_values, text_inputs = dataset_with_template.get_all_data()

In [ ]:
@torch.no_grad()
def extract_image_embeddings(clip_model, image_values, batch_size=32):
    """
    Extracts and L2-normalizes image features.
    """
    device = next(clip_model.parameters()).device
    clip_model.eval()

    img_embs = []
    for i in tqdm(range(0, len(image_values), batch_size), desc="Encoding Images"):
        batch_pixels = image_values[i : i + batch_size].to(device)

        # Extract features
        img_f = clip_model.get_image_features(pixel_values=batch_pixels)
        if not isinstance(img_f, torch.Tensor):
            img_f = img_f.pooler_output

        # L2 Normalization (Crucial for cosine similarity)
        img_f = F.normalize(img_f, p=2, dim=-1)
        img_embs.append(img_f.cpu())

    return torch.cat(img_embs, dim=0)

In [ ]:
@torch.no_grad()
def extract_text_embeddings_from_dict(encoders_dict, text_inputs):
    """
    Iterates through a dictionary of models and extracts normalized
    text embeddings for each.

    Returns:
        dict: { 'checkpoint_name.pt': tensor([Num_Labels, Embedding_Dim]) }
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    all_model_embeddings = {}

    for model_name, model in encoders_dict.items():
        print(f"--- Processing Model: {model_name} ---")
        model.to(device)
        model.eval()

        num_labels = text_inputs['input_ids'].shape[0]
        all_class_features = []

        for i in tqdm(range(num_labels), desc=f"Encoding {model_name}"):
            # Extract templates for this specific label [Num_Templates, Seq_Len]
            label_batch = {k: v[i].to(device) for k, v in text_inputs.items()}

            # Get features for all templates
            template_features = model(label_batch)
            template_features = F.normalize(template_features, p=2, dim=-1)

            # Average templates (Ensemble) and Re-normalize
            class_embedding = template_features.mean(dim=0)
            class_embedding = F.normalize(class_embedding, p=2, dim=0)

            all_class_features.append(class_embedding.cpu())

        # Store the stacked tensor for this specific model
        all_model_embeddings[model_name] = torch.stack(all_class_features, dim=0)

        # Optional: Move model back to CPU to save VRAM if processing many checkpoints
        model.cpu()

    return all_model_embeddings

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score

def compute_zsc_metrics(img_f, txt_f, gt_labels, temperature=0.05, top_k=5):
    """
    Enhanced metrics for ZSC, including Ranking and Class-Balanced stats.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    img_f, txt_f = img_f.to(device), txt_f.to(device)
    gt_tensor = torch.as_tensor(gt_labels).to(device)

    # 1. Similarity & Probs
    # Normalize features for true cosine similarity before logits
    img_norm = F.normalize(img_f, p=2, dim=-1)
    txt_norm = F.normalize(txt_f, p=2, dim=-1)

    logits = (img_norm @ txt_norm.T) / temperature
    probs = F.softmax(logits, dim=1)

    # 2. Accuracy & Ranking
    top_probs, top_indices = torch.topk(probs, k=min(top_k, txt_f.size(0)), dim=1)

    # Top-1 and Top-K
    top1_acc = (top_indices[:, 0] == gt_tensor).float().mean().item()
    topk_acc = (top_indices == gt_tensor.view(-1, 1)).any(dim=1).float().mean().item()

    # Mean Reciprocal Rank (MRR) - Crucial for large synset spaces
    # Finds the rank of the ground truth label in the full sorted list
    ranks = (logits.argsort(dim=1, descending=True) == gt_tensor.view(-1, 1)).nonzero(as_tuple=True)[1]
    mrr = (1.0 / (ranks.float() + 1.0)).mean().item()

    # 3. Macro Metrics (Class Balanced)
    # Using CPU/Numpy for Sklearn metrics
    preds_cpu = top_indices[:, 0].cpu().numpy()
    gt_cpu = gt_tensor.cpu().numpy()

    macro_f1 = f1_score(gt_cpu, preds_cpu, average='macro', zero_division=0)

    # 4. Embedding Stats (The "Health Check")
    # Inter-class similarity: Are your Nepali labels too crowded in vector space?
    class_sim_matrix = txt_norm @ txt_norm.T
    mask = torch.eye(class_sim_matrix.size(0), device=device).bool()
    inter_class_sim = class_sim_matrix[~mask].mean().item()

    return {
        "top1_accuracy": top1_acc,
        "topk_accuracy": topk_acc,
        "mrr": mrr,
        "macro_f1": macro_f1,
        "stats": {
            "avg_inter_class_similarity": inter_class_sim,
            "max_logit": logits.max().item(),
            "avg_confidence": top_probs[:, 0].mean().item(),
            "theoretical_max_sim": class_sim_matrix.max().item()
        }
    }

In [ ]:
def evaluate_all_checkpoints(image_features, embeddings_dict, gt_labels, top_k=5):
    """
    Computes ZSC metrics for every checkpoint in the embeddings dictionary.

    Args:
        image_features: Tensor of shape [Num_Images, Embedding_Dim]
        embeddings_dict: Dict from extract_text_embeddings_from_dict
        gt_labels: List or Tensor of ground truth class indices
    """
    results = {}

    for ckpt_name, text_features in embeddings_dict.items():
        # Compute metrics for this specific checkpoint
        metrics = compute_zsc_metrics(
            img_f=image_features,
            txt_f=text_features,
            gt_labels=gt_labels,
            top_k=top_k
        )

        results[ckpt_name] = metrics

        # Quick print for tracking progress
        t1 = metrics['top1_accuracy'] * 100
        sim = metrics['stats']['avg_inter_class_similarity']
        print(f"[{ckpt_name}] Top-1: {t1:.2f}% | Inter-Class Sim: {sim:.4f}")

    return results

In [ ]:
import re

def get_numeric_step(filename):
    """Extracts numbers from filename for logical sorting (e.g., epoch_10 vs epoch_2)."""
    nums = re.findall(r'\d+', filename)
    return int(nums[0]) if nums else 0

In [ ]:
def log_zsc_report(all_metrics):
    """
    Enhanced ZSC report including MRR and Macro F1 for class-balanced evaluation.
    """
    # Expanded Header for better detail
    header = (f"{'Model Checkpoint':<35} | "
              f"{'Performance (Top1 | Top5 | MRR | F1)':<45} | "
              f"{'Health (Sim | Conf | MaxL)':<35}")

    print("\n" + "=" * len(header))
    print(header)
    print("-" * len(header))

    # Sort models by numeric value in their filename (assuming get_numeric_step exists)
    try:
        sorted_models = sorted(all_metrics.items(), key=lambda x: get_numeric_step(x[0]))
    except NameError:
        sorted_models = sorted(all_metrics.items())

    for model_name, metrics in sorted_models:
        stats = metrics['stats']

        # 1. Performance Metrics (Accuracies + Ranking + Balance)
        # Note: Multiplying by 100 for percentage view
        perf_str = (f"{metrics['top1_accuracy']*100:>5.2f}% | "
                    f"{metrics['topk_accuracy']*100:>5.2f}% | "
                    f"{metrics.get('mrr', 0.0)*100:>5.2f}% | "
                    f"{metrics.get('macro_f1', 0.0)*100:>5.2f}%")

        # 2. Embedding Health Stats
        health_str = (f"Sim: {stats['avg_inter_class_similarity']:>5.3f} | "
                      f"Conf: {stats['avg_confidence']:>5.3f} | "
                      f"MaxL: {stats['max_logit']:>6.2f}")

        print(f"{model_name:<35} | {perf_str:<45} | {health_str:<35}")

    print("=" * len(header) + "\n")

In [ ]:
text_encoders_dict = get_all_models_with_loaded_instance(model_folder=LOCAL_MODEL_PATH,STUDENT_ID=STUDENT_ID)

In [ ]:
image_feats = extract_image_embeddings(clip_model, image_values)
text_feats = extract_text_embeddings_from_dict(text_encoders_dict, text_inputs)

In [ ]:
results = evaluate_all_checkpoints(image_feats, text_feats, gt_labels)

[distillation_epoch_4_step_150000.pt] Top-1: 48.42% | Inter-Class Sim: 0.8472


In [ ]:
log_zsc_report(results)

In [ ]:
results = evaluate_all_checkpoints(image_feats, text_feats, gt_labels)

In [ ]:
log_zsc_report(results)